<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ami-null/python-course/blob/main/10_dataclass.ipynb)

# Python Dataclasses

## Learning Objective
1. Understand what a dataclass is and why it reduces boilerplate
2. Define a dataclass using the `@dataclass` decorator and type hints
3. Use default values and `field()` for scalar and mutable defaults
4. Create immutable dataclasses using `frozen=True`
5. Enable ordering with `order=True` and control comparison fields with `field(compare=False)`
6. Know when a dataclass is appropriate and when a regular class is better

## Introduction

`dataclass` is a decorator in Python that automatically generates special methods for a class based on its annotated attributes. The methods it generates include `__init__`, `__repr__`, and `__eq__`. It is defined in the `dataclasses` module, which is part of the Python standard library.

Dataclasses are particularly useful when a class is meant primarily to hold data rather than implement behavior. Instead of writing the same boilerplate methods repeatedly, you describe the fields with type annotations and Python generates the rest.

Note: decorators are covered in a separate lecture. For now, treat `@dataclass` as a label you place above a class definition to activate the automatic method generation.

## Motivation: Dataclass vs Traditional Class

Consider a simple `Point` class with two fields, `x` and `y`. Using a traditional class, you need to write `__init__` to assign the fields, `__repr__` to get a readable string representation, and `__eq__` to compare two instances by value.

## Defining a Dataclass

To define a dataclass:
1. Import `dataclass` from the `dataclasses` module.
2. Apply `@dataclass` above the class definition.
3. Declare each field as a class-level annotation in the form `field_name: type`.

The order you declare the fields determines the order of parameters in the generated `__init__`.

In [1]:
from dataclasses import dataclass

# Dataclass: all three methods are generated automatically
@dataclass
class Point:
    x: int
    y: int

In [2]:
# The dataclass auto generates __init__, so instantiation works as expected
p1 = Point(2, 3)
p2 = Point(2, 3)
p3 = Point(5, 6)

# __repr__ is also auto generated: produces a readable string
print(p1)

Point(x=2, y=3)


In [3]:
# __eq__ is generated: compares field values, not object identity
print(p1 == p2)   # True: same field values
print(p1 == p3)   # False: different field values

True
False


The following code shows how to define the same class in a traditional way (i.e., without `dataclass`):

In [4]:
# Traditional class: all methods written manually
class PointTraditional:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"PointTraditional(x={self.x}, y={self.y})"

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

In [5]:
# Without @dataclass, __eq__ compares identity by default
pt1 = PointTraditional(2, 3)
pt2 = PointTraditional(2, 3)

# This would return False without a custom __eq__
# because pt1 and pt2 are different objects in memory.
# PointTraditional above has a custom __eq__, so it returns True.
print(pt1 == pt2)

True


Both classes behave the same way, but the dataclass version is significantly shorter. This difference grows as the number of fields increases.

Another complete example:

In [6]:
@dataclass
class Student:
    name: str
    student_id: int
    gpa: float

In [7]:
s1 = Student("Alice", 1001, 3.8)
s2 = Student("Bob", 1002, 3.5)

print(s1)
print(s2)

Student(name='Alice', student_id=1001, gpa=3.8)
Student(name='Bob', student_id=1002, gpa=3.5)


In [8]:
# Accessing fields works the same as any class attribute
print(s1.name)
print(s1.gpa)

Alice
3.8


In [9]:
# __eq__ compares all fields
s3 = Student("Alice", 1001, 3.8)
print(s1 == s3)   # True
print(s1 == s2)   # False

True
False


## Default Values and `field()`

Fields can have default values. A field with a default must come after all fields without defaults, for the same reason as default parameters in regular functions.

For mutable defaults such as lists or dictionaries, you cannot assign the object directly. Python would share the same object across all instances, which causes bugs. Instead, use `field(default_factory=...)` and pass a callable that creates a new object each time.

In [10]:
from dataclasses import dataclass, field

@dataclass
class Student:
    name: str
    student_id: int
    gpa: float = 0.0                          # scalar default
    courses: list = field(default_factory=list)  # mutable default via factory

In [11]:
# gpa and courses use their defaults when not provided
s1 = Student("Alice", 1001)
print(s1)

Student(name='Alice', student_id=1001, gpa=0.0, courses=[])


In [12]:
# Each instance gets its own list: no shared state
s2 = Student("Bob", 1002, 3.5)
s1.courses.append("Math")
s2.courses.append("Physics")

print(s1.courses)   # ['Math']
print(s2.courses)   # ['Physics']

['Math']
['Physics']


In [13]:
# What happens if you try to use a mutable default directly
# UNCOMMENT TO SEE THE ERROR
# @dataclass
# class BadStudent:
#     name: str
#     courses: list = []   # ValueError: mutable default not allowed

## Adding Methods to a Dataclass

Dataclasses can have regular methods just like any other class. The `@dataclass` decorator only generates the special dunder methods; it does not restrict you from adding your own logic.

In [14]:
from dataclasses import dataclass, field

@dataclass
class Student:
    name: str
    student_id: int
    gpa: float = 0.0
    courses: list = field(default_factory=list)

    def letter_grade(self):
        # Returns a letter grade based on the student's GPA
        if self.gpa >= 3.7:
            return "A"
        elif self.gpa >= 3.0:
            return "B"
        elif self.gpa >= 2.0:
            return "C"
        else:
            return "F"

    def enroll(self, course):
        self.courses.append(course)

In [15]:
s1 = Student("Alice", 1001, 3.8)
s1.enroll("Math")
s1.enroll("Physics")

print(s1)
print(s1.letter_grade())

Student(name='Alice', student_id=1001, gpa=3.8, courses=['Math', 'Physics'])
A


## Immutability with `frozen=True`

Passing `frozen=True` to `@dataclass` makes all instances immutable. Attempting to assign to any field after construction raises a `FrozenInstanceError`. This is useful when you want to guarantee that an object's state never changes, for example when using instances as dictionary keys or in sets.

In [16]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Coordinate:
    latitude: float
    longitude: float

In [17]:
c = Coordinate(23.8, 90.4)
print(c)

Coordinate(latitude=23.8, longitude=90.4)


In [18]:
# Attempting to modify a field raises FrozenInstanceError
# UNCOMMENT TO SEE THE ERROR
# c.latitude = 25.0

In [19]:
# frozen=True also generates __hash__, so instances can be used as dictionary keys
locations = {
    Coordinate(23.8, 90.4): "Dhaka",
    Coordinate(48.8, 2.35): "Paris"
}

print(locations[Coordinate(23.8, 90.4)])

Dhaka


## Ordering with `order=True`

By default, dataclasses do not support ordering operators (`<`, `>`, `<=`, `>=`). Passing `order=True` generates these methods by comparing instances field by field in declaration order.

If some fields should not be used for comparison, pass `compare=False` to `field()` for those fields. Only fields with `compare=True` (the default) will be used in the generated ordering and equality methods.

In [20]:
from dataclasses import dataclass, field

@dataclass(order=True)
class Employee:
    salary: float                             # used for ordering
    name: str = field(compare=False)          # excluded from ordering
    department: str = field(compare=False)    # excluded from ordering

In [21]:
employees = [
    Employee(50000, "Alice", "HR"),
    Employee(70000, "Bob", "IT"),
    Employee(60000, "Charlie", "Admin")
]

# sorted() works because order=True generated __lt__ and related methods
sorted_employees = sorted(employees)
print(sorted_employees)

[Employee(salary=50000, name='Alice', department='HR'), Employee(salary=60000, name='Charlie', department='Admin'), Employee(salary=70000, name='Bob', department='IT')]


In [22]:
# Direct comparison also works
e1 = Employee(50000, "Alice", "HR")
e2 = Employee(70000, "Bob", "IT")

print(e1 < e2)    # True: 50000 < 70000
print(e2 > e1)    # True

True
True


## Advantages

1. **Less boilerplate**: No need to manually write `__init__`, `__repr__`, or `__eq__`.
2. **Readable output**: The auto-generated `__repr__` includes field names and values, which is useful during debugging.
3. **Immutability option**: `frozen=True` prevents accidental mutation.
4. **Default values**: Scalar defaults and `default_factory` for mutable types are both supported.
5. **Ordering**: `order=True` generates comparison methods without any extra code.
6. **Type hints**: Field annotations improve readability and enable static type checkers.

## Disadvantages

1. **Not designed for complex behavior**: If a class has significant logic beyond storing data, a regular class is clearer.
2. **Less control over generated methods**: The automatic methods follow fixed rules; overriding them partially can be awkward.
3. **Mutable default pitfall**: Forgetting to use `default_factory` for mutable fields raises a `ValueError` at class definition time.
4. **Slight initialization overhead**: The generated `__init__` is slightly slower than a hand-written one in performance-critical code.

## When Not to Use Dataclasses

- When the class requires significant business logic rather than just holding data.
- When inheritance or a custom `__init__` involves non-trivial setup that the generated `__init__` cannot accommodate.
- When immutability is critical and performance matters; in that case, `namedtuple` or the `attrs` library may be more appropriate.
- When the code needs to run on Python versions before 3.7, since `dataclasses` was introduced in 3.7.

## Exercises

1. Define a `Book` dataclass with fields `title` (str), `author` (str), and `price` (float). Give `price` a default value of `0.0`. Create two `Book` instances and print them. Then compare them using `==` and explain why the result is what it is.

2. Define a frozen dataclass `GPSPin` with fields `latitude` (float) and `longitude` (float). Create an instance, then try to modify one of its fields. Comment out the offending line with `# UNCOMMENT TO SEE THE ERROR`. Then create two `GPSPin` instances with the same coordinates and verify they compare equal.

3. Extend the `Student` dataclass from the lecture (with `name`, `student_id`, `gpa`, and `courses`) by adding a method `summary()` that returns a formatted string such as `"Alice (ID: 1001) | GPA: 3.8 | Courses: Math, Physics"`. If the student has no courses, the string should end with `"Courses: none"`. Test it with at least two instances.

4. Create a `Product` dataclass with fields `price` (float) and `name` (str), with ordering enabled so that products can be sorted by price. Use `field(compare=False)` appropriately. Build a list of at least four products and print the sorted list. Also verify that direct comparison with `<` works between two products.

5. Below is a traditional class. Convert it to a dataclass and note which methods you were able to remove.

```python
class Movie:
    def __init__(self, title, director, year, rating=0.0):
        self.title = title
        self.director = director
        self.year = year
        self.rating = rating

    def __repr__(self):
        return f"Movie(title={self.title!r}, director={self.director!r}, year={self.year}, rating={self.rating})"

    def __eq__(self, other):
        return (self.title == other.title and
                self.director == other.director and
                self.year == other.year and
                self.rating == other.rating)
```